# Cleaning the dataset 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import re
import string
import time

pd.set_option('display.max_columns', 50)

### Read in Raw Dataset

In [ ]:
# reading in fixed metadata

# set data types
cols = ['PROJECT_ID', 'ABSTRACT', 'FY.x', 'PROJECT_TERMS', 'PROJECT_TITLE', 'DEPARTMENT', 'AGENCY', 'IC_CENTER', 
        'PROJECT_NUMBER', 'PROJECT_START_DATE', 'PROJECT_END_DATE', 'CONTACT_PI_PROJECT_LEADER', 'OTHER_PIS', 
        'CONGRESSIONAL_DISTRICT', 'DUNS_NUMBER', 'ORGANIZATION_NAME', 'ORGANIZATION_CITY', 'ORGANIZATION_STATE', 
        'ORGANIZATION_ZIP', 'ORGANIZATION_COUNTRY', 'BUDGET_START_DATE', 'BUDGET_END_DATE', 'CFDA_CODE', 'FY.y', 
        'FY_TOTAL_COST', 'FY_TOTAL_COST_SUB_PROJECTS']
dtypes = {col: 'str' for col in cols}
dtypes["FY_TOTAL_COST"] = 'float'
dtypes["FY_TOTAL_COST_SUB_PROJECTS"] = 'float'

df = pd.read_csv('../../../data/prd/Federal_RePORTER/FR-2021DEC/FR_raw_2021DEC17.csv', dtype = dtypes, engine='python')
print(df.shape)

In [ ]:
df.head()

#### some brief eda

In [ ]:
len(pd.unique(df["PROJECT_ID"]))

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
temp = df[df['FY_TOTAL_COST'].isnull()]
len(temp)

In [ ]:
temp['DEPARTMENT'].value_counts()

In [ ]:
# are FY.x and FY.y equal?  YES except for the entries where FY.x is NULL

temp = df[df["FY.x"] != df["FY.y"]]

In [ ]:
print(len(temp))
temp['FY.x'].isnull().sum()

# CONCLUSION - Use FY.y for FY, drop FY.x

In [ ]:
# FY.y is the reliable fiscal year information so we rename this column to FY

df = df.rename(columns={'FY.y': 'FY'})

In [ ]:
df = df.drop(columns = ['FY.x'])

### Remove null abstracts

In [ ]:
#remove rows with NULL abstracts

l1 = len(df)
df = df[~df.ABSTRACT.isnull()]
l2 = len(df)

print(l1-l2, "null ABSTRACTs removed")

### Aggregate Counts

In [ ]:
# strip white space from character columns

for col in df.columns[0:23]:
    print(col)
    df[col] = df[col].str.strip()

**Aggregate funding amounts for rows with same ABSTRACT/TITLE/FY**

In [ ]:
# Checking into duplicates first to examine FY_TOTAL_COST 

dup = df[df.duplicated(subset=['ABSTRACT',  'PROJECT_TITLE', 'FY'], keep=False)]
print(len(dup))

In [ ]:
dup.head()

In [ ]:
# look at specific dupicate group examples

#dup[dup['ABSTRACT'] == dup["ABSTRACT"].iloc[0]]

temp = df[df['PROJECT_TITLE'] == 'REDUCING HOSPITAL READMISSION RATES BY IMPLEMENTING AN INPATIENT TOBACCO CESSATION SERVICE DRIVEN BY INTERACTIVE-VOICE RECOGNITION TECHNOLOGY'] 
temp

# TAKEAWAY - Abstracts are basically the same for this example.  Differs in a junk phrase and the start and minor
# punctuation, capitalization.  If FY had been the same though, these wouldn't have been counted as duplicates.

In [ ]:
# OVERALL TAKEAWAY - sum total cost for each group of duplicates

**Aggregate counts for unique ORGANIZATION_NAMEs, PIs, and PROJECT_IDs in rows with same Abstract/Title/FY**

In [ ]:
# group df by abstract/title/FY (exact matches)
all_grp = df.groupby(['ABSTRACT', 'PROJECT_TITLE', 'FY'], dropna=False)

In [ ]:
# for each unique ABSTRACT/TITLE/FY in df, count unique Organizations, unique PIs, and group size
agg_df = all_grp.agg(ORG_COUNT=('ORGANIZATION_NAME', 'nunique'),
                     PI_COUNT=('CONTACT_PI_PROJECT_LEADER', 'nunique'),
                     FY_TOTAL_COST_SUM=('FY_TOTAL_COST', 'sum'),
                     NUM_RECORDS=('PROJECT_ID', 'count')
                    )

# merge to bring in the aggregated information
merged = df.merge(agg_df, how='left', on=['ABSTRACT', 'PROJECT_TITLE', 'FY'])

In [ ]:
agg_df.head()

In [ ]:
agg_df['NUM_RECORDS'].sum()

In [ ]:
len(df)

In [ ]:
merged.head()

### Deduplication

In [ ]:
print(sum(merged['ABSTRACT'].isnull()))
print(sum(merged['PROJECT_TITLE'].isnull()))
print(sum(merged['FY'].isnull()))

In [ ]:
# save NON-duplicated rows and the LAST occurrance of duplicated rows
dedup = merged[~merged.duplicated(subset=['ABSTRACT',  'PROJECT_TITLE', 'FY'], keep='last')]

In [ ]:
print(f"Original: {len(df)}")
print(f"Deduplicated: {len(dedup)}")
print(f"Number of Duplicates: {len(df) - len(dedup)}")

In [ ]:
dedup['NUM_RECORDS'].sum()

In [ ]:
#save final (deduplicated) dataframe as "df" to fit downstream code
df = dedup

### Dataset setup

##### Helper function used throughout

In [ ]:
# drops strings with length 0 - indices printed out, find with df.loc[ix]

def drop_empties(df, col):
    
    l1 = len(df)
    ix = df[df[col].apply(len)==0].index
    print(ix)
    df.drop(ix,axis=0,inplace=True)
    l2 = len(df)
    
    print(f"dropped {l1-l2}")
    
    return df

In [ ]:
# strip leading and trailing whitespace, save in a working abstract column that will be updated as text is cleaned
# strip() already ran above

# Note: we cannot lower case abstracts up front - capitalization is needed to find POS in preprocessing
 
#df = df.assign(working_abstract = [abstract.strip() for abstract in df["ABSTRACT"]])
df["working_abstract"] = df["ABSTRACT"]
df = drop_empties(df, "working_abstract")

wa = 'working_abstract'

In [ ]:
def remove_short_abstracts(df, limit):

    # Remove abstracts with length < limit. 150 seems like a good cutoff, but it does lose some useful information.
    
    # what do we want to do for the cutoff?  -- SOLUTION UNTIL FURTHER EXPLORATION -> keep the same
    
    df['nchar']=df['working_abstract'].apply(len)
    l1 = len(df)
    df=df.loc[df['nchar']>=limit]
    l2 = len(df)
    
    print(l1-l2, "short abstracts removed")
    
    return df

df = remove_short_abstracts(df,limit=150)

In [ ]:
len(df)

### Cleaning Strategy:
1. Remove abstracts with all non-alphanumeric characters.
2. Remove non-alphanumeric characters from the start and end of abstracts
3. Remove other non-readable abstracts. (REMOVAL ABSTRACTS FOUND BY INSPECTION)
4. Remove "junk" starting strings and ending strings
5. Remove "junk" strings in the middle 


##### Helper functions for cleaning

In [ ]:
def strip_nonalnum(word):
    
    # function strips non-alphanumeric characters from the beginning and end of a string
    # adapted from: https://stackoverflow.com/questions/22650506/how-to-rermove-non-alphanumeric-characters-at-the-beginning-or-end-of-a-string
        
    if not word:
        return word  # nothing to strip
    if (len(word) == 1) and (not word[0].isalnum()):  
        return ""
    for start, c in enumerate(word):
        if c.isalnum():
            break
    for end, c in enumerate(word[::-1]):
        if c.isalnum():
            break
                  
    return word[start:len(word) - end]


In [ ]:
def remove_phrase(x, phrase,loc='Start'):
    
    # returns x with phrase removed. location can be "Start" of string, "End" of string, or 
    # "Anywhere_All"--anywhere will remove all instances and Anywhere_First will remove the first instance
    # CASE info - this function assumes phrase is lower case, but that x is not.
    
    assert loc in ['Start','End']
    
    if loc=='End':
        if x.lower().endswith(phrase):
            return x[:-1*len(phrase)].strip()
        else:
            return x
    elif loc=='Start':
        if x.lower().startswith(phrase):
            return x[len(phrase):].strip()
        else:
            return x
    else:
        return 'Error'
    

In [ ]:
# strip non-alphanum characters from the beginning and end of each abstract

temp = [strip_nonalnum(abstract) for abstract in df["working_abstract"]]
df = df.assign(working_abstract = temp)

df = drop_empties(df, "working_abstract")

In [ ]:
# remove abstracts that are not readable: THIS NEEDS TO BE UPDATED BY HAND FOR EVERY NEW DATASET
# For example: index = 883794: ? ?/? ? ? ? ? ? ? ? ...

df = df.assign(Start_Char = df['working_abstract'].apply(lambda x: x[0]))
ix = df[df['Start_Char'] == '?'].index
print(ix)

In [ ]:
df.drop(index = ix, inplace = True)

In [ ]:
# "junk" phrases at start to remove

start_phrases=['abstract', 'summary', 'proposal', 'description', 'narrative', 
               'technical abstract',
               'non technical abstract', 
               'non- technical abstract',
               'non-technical abstract',                      
               'nontechnical abstract',
               'technical summary', 
               'nontechnical summary',
               'non-technical summary',
               'non-technical description',
               'description (provided by the applicant)',
               'description (provided by investigator)',  
               'description (provided by applicant)',
               'project summary/abstract',
               'proposal abstract',
               'research abstract',
               'project summary',
               'research summary',
               'project description'
               'see instructions):',
               'for center application (provided by the investigator):',
               'objective(s)',      
               'exceed the space provided',
               'provided by applicant',
               'provided by candidate']


In [ ]:
#Remove found start phrases

for phrase in start_phrases:
    temp = df[wa].apply(remove_phrase,args=[phrase,'Start'])
    df = df.assign(working_abstract = temp) 

# strip non-alphanum characters from the beginning and end of each abstract

temp = [strip_nonalnum(abstract) for abstract in df["working_abstract"]]
df = df.assign(working_abstract = temp)

df = drop_empties(df, "working_abstract")

In [ ]:
#Repeated start phrase removal in case the order of project summary/abstract varies

for phrase in start_phrases:
    temp = df[wa].apply(remove_phrase,args=[phrase,'Start'])
    df = df.assign(working_abstract = temp)  
    
df = drop_empties(df, "working_abstract")

In [ ]:
#starting_exact_phrases to remove

#'This subproject represents an estimate of the percentage of the CTSA funding that isbeing utilized for a broad area of research (AIDS research, pediatric research, orclinical trials).  The Total Cost listed is only an estimate of the amount of CTSAinfrastructure going towards this area of research, not direct funding provided bythe NCRR grant to the subproject or subproject staff.'
#'This subproject is one of many research subprojects utilizing theresources provided by a Center grant funded by NIH/NCRR. The subproject andinvestigator (PI) may have received primary funding from another NIH source,and thus could be represented in other CRISP entries. The institution listed isfor the Center, which is not necessarily the institution for the investigator.'

temp = df[wa].apply(lambda x: x.replace('This subproject represents an estimate of the percentage of the CTSA funding that isbeing utilized for a broad area of research (AIDS research, pediatric research, orclinical trials).  The Total Cost listed is only an estimate of the amount of CTSAinfrastructure going towards this area of research, not direct funding provided bythe NCRR grant to the subproject or subproject staff.',
                                       ''))
df = df.assign(working_abstract = temp) 

expression=re.compile('This subproject is one of many research subprojects.*not necessarily the institution for the investigator.')
temp = df[wa].apply(lambda x: re.sub(expression,'',x))
df = df.assign(working_abstract = temp)

expression=re.compile('This subproject is one of many research subprojects.*to the subproject or subproject staff.')
temp = df[wa].apply(lambda x: re.sub(expression,'',x))
df = df.assign(working_abstract = temp)

In [ ]:
# strip non-alphanum characters from the beginning and end of each abstract

temp = [strip_nonalnum(abstract) for abstract in df["working_abstract"]]
df = df.assign(working_abstract = temp)

df = drop_empties(df, "working_abstract")

In [ ]:
end_phrases = ['(end of abstract',
               'end of abstract', 
               '(abstract end',  
               '(end of abstract',
               '(end 0f abstract',
               '(end of absract',
               '(abstract below',
               '(end of reviewers\' comment',
               '(end abstract',
               'performance site ========================================section end',
               'key personnel ========================================section end',
               '[summary truncated at 7800 characters', 
               'this award reflects nsf\'s statutory mission and has been deemed worthy of support through evaluation using the foundation\'s intellectual merit and broader impacts review criteria',
               'project description page 6', 'page 1 of 1', 'project summary/abstract page 6',
               'project description page 7', 'project summary/abstract page 7', 'pag 1 o 1', 
               'page 2 number pages consecutively at the bottom throughout form page 2']


In [ ]:
# end phrase removal

for phrase in end_phrases:
    temp = df[wa].apply(remove_phrase,args=[phrase,'End'])
    df = df.assign(working_abstract = temp) 

# strip non-alphanum characters from the beginning and end of each abstract

temp = [strip_nonalnum(abstract) for abstract in df["working_abstract"]]
df = df.assign(working_abstract = temp)

df = drop_empties(df, "working_abstract")

In [ ]:
# "junk" removal within text body - not necessarily at the start or end

# 'Enter the text here that' ending with 'lines of text.'
expression=re.compile('Enter the text here that.*lines of text')
temp=df[wa].apply(lambda x: re.sub(expression,'',x))
df = df.assign(working_abstract = temp)

expression=re.compile('PHS .*?Continuation Format Page')
temp=df[wa].apply(lambda x: re.sub(expression,'',x))
df = df.assign(working_abstract = temp)

expression=re.compile('OMB No .*?Continuation Format Page')
temp=df[wa].apply(lambda x: re.sub(expression,'',x))
df = df.assign(working_abstract = temp)

temp=df[wa].replace('Project Summary/Abstract','')
df = df.assign(working_abstract = temp)

In [ ]:
"""If it starts with 'one page and must contain',
This is an NIH thing and there aren't that many of them, but come from 3 different cfda
it will start with "one page and must contain a summary of the proposed activity suitable for dissemination to 
thepublic. It should be a self-contained description of the project and should contain a statement of objectives 
and methods to be employed. It should be informative to other persons working in the same or related fields and 
insofar as possible understandable to a technically liter-ate lay reader. This Abstract must not include any 
proprietary/confidential information.* Please click the add attachment button to complete this entry." plus some 
attachments, which includes tracking number, twice: following the second trackign number, there is a grant number
followed by the actual content" 

At the end of these files, they all end in 'Project Narrative File'(last instance) followed by more attachments, 
all of which can be discarded
"""

expression1=re.compile('one page and must.*?Tracking Number.*?(Tracking Number)')
expression2=re.compile('Project Narrative File.*')

def fix_abstract(abstract):
    if abstract.startswith('one page and must contain'):
        abstract=re.sub(expression1,'',abstract)
        return re.sub(expression2,'',abstract)
    else:
        return abstract

temp=df[wa].apply(fix_abstract)
df = df.assign(working_abstract = temp)

In [ ]:
# removal of phrase at end

expression=re.compile('Project Summary/Abstract Page.*')

def remove_contact_pd(x):
    
    """removes clause at end that tends to occur: eg Project Summary/Abstract Page 222Contact PD/PI: Sampson, HughNarrative ("""
    
    if x.startswith('Contact PD/PI'):
        return re.sub(expression,'',x)
    else:
        return x
    
temp=df[wa].apply(remove_contact_pd) 
df = df.assign(working_abstract = temp)

In [ ]:
# strip non-alphanum characters from the beginning and end of each abstract

temp = [strip_nonalnum(abstract) for abstract in df["working_abstract"]]
df = df.assign(working_abstract = temp)

df = drop_empties(df, "working_abstract")

In [ ]:
len(df)

In [ ]:
df.to_pickle("../../../data/prd/Paper/FR_clean_22DEC21.pkl")
